# ESM-2 Tutorial: Embeddings and Fine-Tuning on GFP

**ESM-2** (Evolutionary Scale Modeling 2) is a protein language model from Meta AI, pre-trained on hundreds of millions of protein sequences. This notebook shows you how to:

1. Load ESM-2 as a frozen embedding extractor
2. Extract per-sequence embeddings for GFP (Green Fluorescent Protein) variants
3. Visualise the embedding space with UMAP, coloured by fluorescence
4. Train a regression head on top of frozen embeddings
5. Use `predict()` on the trained model

### Environment Setup

From the `/tutorials` directory:
```
uv sync
source .venv/bin/activate
```
Select `.venv` as the kernel when prompted.

**Expected runtime:** < 5 minutes on CPU using `facebook/esm2_t6_8M_UR50D` (8M parameters, ~31 MB download on first run).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
import umap
from alf_core import BaseDatasetConfig, LabelledCandidates, Modality, ProblemType
from alf_tools.datasets.gfp import GFP
from alf_tools.models.esm2 import ESM2Model, ESM2ModelConfig, ESM2TrainConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 1. Load the GFP Dataset

**Green Fluorescent Protein (GFP)** is a classic protein engineering benchmark. The dataset contains ~1 000 nucleotide sequences encoding GFP variants, each labelled with `medianBrightness` — a proxy for how well the variant fluoresces.

`GFP.load_dataset()` downloads and caches the CSV on first call, then returns a `LabelledCandidates` object where each `Candidate.data` is a nucleotide sequence string and `.labels` is a `numpy` array of brightness scores.

We use 50 sequences for embedding and fine-tuning to keep CPU runtime under 5 minutes.

> **Note on sequence type:** ESM-2 is a protein language model trained on amino acid sequences. The GFP dataset provides nucleotide (DNA) sequences, but since the characters A/C/G/T are valid amino acid single-letter codes (Ala/Cys/Gly/Thr), the tokenizer accepts them. This tutorial focuses on demonstrating the ALF API; in production use you would translate codons to amino acid sequences before embedding with ESM-2.

In [ ]:
gfp_config = BaseDatasetConfig(
    name="gfp",
    modality=Modality.SEQUENCE,
    problem_type=ProblemType.REGRESSION,
    seed=42,
    train_ratio=0.8,
    test_ratio=0.1,
    validation_frac=0.1,
)
gfp = GFP(gfp_config)
data: LabelledCandidates = gfp.load_dataset()

print(f"Total sequences : {len(data.candidates)}")
print(f"Example sequence: {data.candidates[0].data[:40]}...")
print(f"Brightness range: {data.labels.min():.2f} – {data.labels.max():.2f}")

In [ ]:
embed_candidates = data.candidates[:50]
embed_labels = data.labels[:50]  # numpy array, shape (50,)

finetune_candidates = data.candidates[:50]
finetune_labels = data.labels[:50]  # numpy array, shape (50,)

## 2. Initialise ESM-2 as a Frozen Embedding Extractor

**`ESM2ModelConfig`** controls the architecture:
- `model_id` — any `facebook/esm2_*` HuggingFace checkpoint
- `pooling` — how to collapse per-token hidden states to one vector: `"mean"` (average over all non-padding positions, CLS and EOS included), `"cls"` (first token), or `"last_hidden_state"` (full sequence tensor)
- `repr_layer` — which transformer layer to read; `-1` is the final layer

**`ESM2TrainConfig(scoring_function=None)`** disables the linear prediction head. We only call `embed()` on this model, so `scoring_function=None` makes it explicit that no prediction head is attached. To enable supervised prediction, pass `scoring_function="linear_head"` — covered in Section 4.

> **Contributors:** `ESM2Model.embed()` in `tools/alf_tools/models/esm2.py` tokenizes via `featurise()`, runs a forward pass with `output_hidden_states=True`, then pools the selected hidden layer.

In [ ]:
model_cfg = ESM2ModelConfig(
    model_id="facebook/esm2_t6_8M_UR50D",
    pooling="mean",
    repr_layer=-1,
)
train_cfg = ESM2TrainConfig(scoring_function=None)

model = ESM2Model(name="esm2-gfp", model_config=model_cfg, train_config=train_cfg, device=device)
print(model)

## 3. Extract Sequence Embeddings

`embed()` runs a batched forward pass and returns a `(N, hidden_dim)` numpy array — one 320-dimensional vector per sequence.

Note: `featurise()` only tokenizes (returns `{"input_ids", "attention_mask"}`); the forward pass and pooling live entirely in `embed()`.

In [ ]:
X = model.embed(embed_candidates)  # shape (50, 320)
print(f"Embedding shape: {X.shape}")

## 4. UMAP of Frozen Embeddings

We project 320-dimensional embeddings to 2D with UMAP and colour each point by fluorescence. If ESM-2 has captured meaningful structure in GFP sequence space, high-brightness variants should cluster rather than scatter uniformly.

In [ ]:
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
coords = reducer.fit_transform(X)

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(coords[:, 0], coords[:, 1], c=embed_labels, cmap="viridis", s=15, alpha=0.8)
plt.colorbar(sc, ax=ax, label="Median Brightness")
ax.set_title("UMAP of ESM-2 embeddings (frozen) — GFP sequences")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
plt.tight_layout()
plt.show()

## 5. Train a Linear Regression Head

For supervised fitness prediction, set `scoring_function="linear_head"` in `ESM2TrainConfig`. This freezes the ESM-2 backbone and trains a small linear head on top of pooled sequence embeddings using labelled fitness data.

We split our 50 sequences into 40 train / 10 val and train for 2 epochs — enough to see the loss decrease on CPU in under 2 minutes.

> `loss_fn="mse"` with `output_dim=1` configures a regression head. For classification, use `loss_fn="cross_entropy"` and set `output_dim` to the number of classes.

In [ ]:
n_train = 40
train_data = LabelledCandidates(
    candidates=finetune_candidates[:n_train],
    labels=finetune_labels[:n_train],
)
val_data = LabelledCandidates(
    candidates=finetune_candidates[n_train:],
    labels=finetune_labels[n_train:],
)

In [ ]:
train_cfg_reg = ESM2TrainConfig(
    scoring_function="linear_head",
    loss_fn="mse",
    output_dim=1,
    num_epochs=2,
    batch_size=4,
    learning_rate=1e-3,
)
model_reg = ESM2Model(
    name="esm2-gfp-reg", model_config=model_cfg, train_config=train_cfg_reg, device=device
)
model_reg.train(train_data, val_data)

metrics_reg = model_reg.get_training_summary_metrics()
print("Linear head summary metrics:", metrics_reg)

In [ ]:
val_preds = model_reg.predict(finetune_candidates[n_train:])

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(finetune_labels[n_train:], val_preds.means, alpha=0.8, s=40)
ax.axline((0, 0), slope=1, color="gray", linestyle="--", linewidth=1, label="y = x")
ax.set_xlabel("True Brightness")
ax.set_ylabel("Predicted Brightness")
ax.set_title("Validation predictions — linear head on ESM-2")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
metrics_df = pd.DataFrame([metrics_reg], index=["linear head"])
display(metrics_df)

## 6. Predict with the Trained Model

`model_reg.predict()` returns regression values from the trained linear head, wrapped in a `Predictions` object whose `.means` is a `(N,)` numpy array. `.variances` is always `None` for ESM-2.

In [ ]:
test_candidates = data.candidates[200:210]

reg_preds = model_reg.predict(test_candidates)
print(f"Predictions shape  : {reg_preds.means.shape}")  # (10,)
print(f"First 3 predictions: {reg_preds.means[:3]}")
print(f"Variances          : {reg_preds.variances}")  # None